<p align="center">
  <img src="../assets/prodinno_logo.png" alt="Prodinno" width="200">
</p>

<h4 align="center">Session 2 · XGBoost & Random Forest</h4>
<h1 align="center">Random Forest — Hotel Booking Demand</h1>
<p align="center"><i>Removing leakage, fixing missing values, and encoding features for a tree-based model</i></p>

---


## 1. What this notebook does

This notebook turns the deduplicated dataset from `00_dataset_and_impurity.ipynb` into a
model-ready train/test split. Four jobs, in order:

1. **Remove target leakage** — and prove, with a quick throwaway model, why it matters.
2. **Handle missing values** column by column, with a reason for each choice.
3. **Feature-engineer** two new columns.
4. **Encode categoricals** appropriately for a tree-based model, and split into train/test.

We deliberately do **not** scale any numeric features here — the reason is explained in
Section 5.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42

df = pd.read_csv("data/hotel_bookings_clean.csv")
print(df.shape)
df.head(3)

(87396, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02



## 2. The leakage lesson: `reservation_status` and `reservation_status_date`

This is one of the most common real-world mistakes in applied machine learning, and it is
worth seeing it happen before we simply tell you to avoid it.

Look at the `reservation_status` column: its values are `"Canceled"`, `"Check-Out"`, and
`"No-Show"`. Now compare that to our target, `is_canceled` (0/1). These two columns are, for
all practical purposes, **the same piece of information written in two different formats** —
`reservation_status == "Canceled"` is (almost) exactly `is_canceled == 1`. And
`reservation_status_date` — the date the status was last updated — is a fact that, for a
canceled booking, is **only known after the cancellation already happened**. Neither column
could possibly be known at the time we'd actually want to make a prediction (i.e. when the
booking is made, before we know the outcome).

Let's prove this with a quick, throwaway model that we will then discard.


In [2]:
leak_check = df.copy()

# Minimal encoding just for this throwaway demonstration
leak_check["reservation_status"] = leak_check["reservation_status"].astype("category").cat.codes
leak_check["reservation_status_date"] = pd.to_datetime(leak_check["reservation_status_date"]).astype("int64")

# Use only numeric columns plus the two suspect ones, dropping obviously non-numeric text columns for simplicity
demo_features = leak_check.select_dtypes(include=[np.number]).drop(columns=["is_canceled"]).columns.tolist()

X_demo = leak_check[demo_features].fillna(-1)
y_demo = leak_check["is_canceled"]

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_demo, y_demo, test_size=0.2, random_state=RANDOM_STATE, stratify=y_demo
)

leaky_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
leaky_model.fit(X_train_d, y_train_d)
leaky_acc = accuracy_score(y_test_d, leaky_model.predict(X_test_d))
print(f"Test accuracy WITH reservation_status / reservation_status_date included: {leaky_acc:.4%}")

Test accuracy WITH reservation_status / reservation_status_date included: 100.0000%



That accuracy is *suspiciously* close to perfect. A real hotel-cancellation model, predicting
before the fact, has no business being anywhere near that good — cancellation depends on a
guest's future behavior, which is inherently uncertain. **When a model looks too good to be
true on a hard real-world prediction problem, the first thing to check is whether the outcome
leaked into the inputs.** Here it obviously did: `reservation_status` essentially *encodes*
`is_canceled`.

Now let's drop both columns and see what a more honest baseline looks like.


In [3]:
X_demo_clean = X_demo.drop(columns=[c for c in ["reservation_status", "reservation_status_date"] if c in X_demo.columns])

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_demo_clean, y_demo, test_size=0.2, random_state=RANDOM_STATE, stratify=y_demo
)

honest_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
honest_model.fit(X_train_c, y_train_c)
honest_acc = accuracy_score(y_test_c, honest_model.predict(X_test_c))
print(f"Test accuracy WITHOUT the leaky columns: {honest_acc:.4%}")
print(f"Drop in accuracy from removing leakage: {(leaky_acc - honest_acc):.4%} points")

Test accuracy WITHOUT the leaky columns: 81.3558%
Drop in accuracy from removing leakage: 18.6442% points



The accuracy drops substantially once the leaky columns are removed — and this lower number
is the honest one. It is a harder, more realistic problem, and that's the point: **a model
that "cheats" by seeing the outcome disguised as an input will always look better than a
model that has to actually predict the future.**

> **Rule to internalize:** for every feature, ask "would this value actually be known and
> available at the moment we need to make the prediction?" If the answer is no — because it
> is a restatement of the outcome, or because it's only recorded after the outcome occurs —
> drop it, no matter how much it improves your metrics. Especially because it improves your
> metrics.

We now proceed with the real, leakage-free pipeline. The `leak_check`, `leaky_model`, and
`honest_model` objects above were purely for this demonstration and are not reused below.


In [4]:
df = df.drop(columns=["reservation_status", "reservation_status_date"])
print(f"Shape after dropping leakage columns: {df.shape}")

Shape after dropping leakage columns: (87396, 30)



## 3. Handling missing values

Recall from `00_dataset_and_impurity.ipynb` that four columns carry missing values, each for a
different underlying reason. We handle each on its own terms rather than applying one
blanket rule.


In [5]:
missing_before = df.isna().sum()
missing_before[missing_before > 0]

children        4
country       452
agent       12193
company     82137
dtype: int64


- **`company`** — missing in ~94% of rows. This isn't usably imputable at that missingness
  rate; any value we invent is mostly noise, and the column would swamp more informative
  features with junk. We **drop the column** entirely.
- **`agent`** — missing in ~13.7% of rows, most plausibly because *no travel agent was
  involved* in that booking rather than a recording failure. Filling it with a *numeric*
  value (like the mean agent ID, or 0) would be nonsensical — agent IDs are categorical codes,
  not a quantity. Instead we cast `agent` to string and fill missing values with the sentinel
  category `"None"`, which explicitly represents "no agent," and let it get frequency-encoded
  like every other agent ID.
- **`country`** — missing in <1% of rows. We fill with the explicit category `"Unknown"`
  rather than guessing a country.
- **`children`** — only 4 rows missing out of 87,396. We fill with `0`, both the median and
  the mode for this heavily-zero-skewed count column.


In [6]:
df = df.drop(columns=["company"])

df["agent"] = df["agent"].astype("Int64").astype(str).replace("<NA>", "None")
df.loc[df["agent"] == "nan", "agent"] = "None"

df["country"] = df["country"].fillna("Unknown")

df["children"] = df["children"].fillna(0)

print("Remaining missing values:")
print(df.isna().sum().sum())
df[["agent", "country", "children"]].head(5)

Remaining missing values:
12193


,agent,country,children
0,NaN,PRT,0.0
1,NaN,PRT,0.0
2,NaN,GBR,0.0
3,304,GBR,0.0
4,240,GBR,0.0



All missing values have now been resolved with a deliberate, explainable choice rather than a
one-size-fits-all rule. Zero `NaN`s remain in the dataframe.



## 4. Feature engineering

We add two small but genuinely useful engineered features:

- **`total_nights`** $= \text{stays\_in\_weekend\_nights} + \text{stays\_in\_week\_nights}$ —
  total length of stay, which is easier for a tree to split on directly than reasoning about
  two separate night counts.
- **`has_children_or_babies`** $= \mathbb{1}[(\text{children} + \text{babies}) > 0]$ — a
  simple binary flag for "traveling with young dependents," which may relate to trip type
  (family vs. business/solo) and therefore to cancellation behavior.


In [7]:
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["has_children_or_babies"] = ((df["children"] + df["babies"]) > 0).astype(int)

df[["stays_in_weekend_nights", "stays_in_week_nights", "total_nights",
    "children", "babies", "has_children_or_babies"]].head(5)

,stays_in_weekend_nights,stays_in_week_nights,total_nights,children,babies,has_children_or_babies
0,0,0,0,0.0,0,0
1,0,0,0,0.0,0,0
2,0,1,1,0.0,0,0
3,0,1,1,0.0,0,0
4,0,2,2,0.0,0,0



## 5. Encoding categoricals — and why we skip scaling

### No feature scaling needed

Unlike logistic regression (where feature scale directly affects the optimizer's convergence
and the coefficients' comparability), **decision trees and Random Forests are invariant to
monotonic transformations of a single feature.** A tree split like
`lead_time <= 87.5` finds exactly the same partition of the data whether `lead_time` is
measured in days, hours, or z-scores — the split threshold just moves proportionally. Since
Random Forest only ever asks "is this feature above or below some threshold?", rescaling a
feature cannot change which rows end up on which side of any split. We therefore **skip
`StandardScaler`/`MinMaxScaler` entirely** — it would add a preprocessing step for zero
benefit.

### One-hot encoding for moderate-cardinality categoricals

For categorical columns with a small, fixed number of categories, we use one-hot encoding —
each category becomes its own 0/1 column. This is appropriate for:
`hotel`, `meal`, `market_segment`, `distribution_channel`, `reserved_room_type`,
`assigned_room_type`, `deposit_type`, `customer_type`. We also one-hot encode
`arrival_date_month` here (12 fixed categories) — note that one-hot encoding does not care
about calendar ordering the way a plot does, since each month becomes an independent 0/1
column and a tree can combine them freely, so the alphabetical-vs-calendar ordering issue we
flagged in the EDA notebook is purely a *visualization* pitfall, not an *encoding* one.


In [8]:
onehot_cols = ["hotel", "meal", "market_segment", "distribution_channel",
               "reserved_room_type", "assigned_room_type", "deposit_type", "customer_type",
               "arrival_date_month"]

for col in onehot_cols:
    print(f"{col:22s}: {df[col].nunique()} categories")

hotel                 : 2 categories
meal                  : 5 categories
market_segment        : 8 categories
distribution_channel  : 5 categories
reserved_room_type    : 10 categories
assigned_room_type    : 12 categories
deposit_type          : 3 categories
customer_type         : 4 categories
arrival_date_month    : 12 categories



Each of these columns has somewhere between 2 and roughly a dozen categories — one-hot
encoding them adds a manageable number of new columns.

### Frequency encoding for high-cardinality categoricals

`country` (178 unique values) and `agent` (now including the `"None"` sentinel, ~334 unique
values including it) are a different story. One-hot encoding either would create hundreds of
mostly-empty binary columns — most of them 1 for a handful of rows and 0 everywhere else. That:

- massively inflates dimensionality for very little signal per column,
- dilutes the "random feature subsampling" mechanism Random Forest relies on (with hundreds
  of near-useless sparse columns, a random subset of features is less likely to contain a
  genuinely useful split candidate at any given node),
- and mostly just re-encodes "how common is this category," which we can capture far more
  compactly.

Instead we use **frequency encoding**: replace each category with how often it appears in the
training data. This keeps the column count small while still letting the tree distinguish
"a country/agent we see constantly" from "a country/agent we've barely seen," which is
frequently exactly the useful signal buried in a high-cardinality column.


In [9]:
print(f"country unique values: {df['country'].nunique()}")
print(f"agent unique values:   {df['agent'].nunique()}")

country unique values: 178
agent unique values:   333



## 6. Train/test split (stratified)

We split **before** fitting the frequency encoding, so that the encoding is learned only from
the training data and applied to the test data — this avoids leaking test-set frequency
information into the training features. The split is **stratified on `is_canceled`** so both
sets preserve the same ~27.5% cancellation rate.


In [10]:
y = df["is_canceled"]
X = df.drop(columns=["is_canceled"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train shape: {X_train.shape}   Train cancellation rate: {y_train.mean():.4%}")
print(f"Test  shape: {X_test.shape}   Test  cancellation rate: {y_test.mean():.4%}")

Train shape: (69916, 30)   Train cancellation rate: 27.4901%
Test  shape: (17480, 30)   Test  cancellation rate: 27.4886%


In [11]:
# Frequency encoding, fit on train only, applied to both
freq_cols = ["country", "agent"]

for col in freq_cols:
    freq_map = X_train[col].value_counts(normalize=True)
    X_train[col + "_freq"] = X_train[col].map(freq_map)
    # Unseen categories in test -> frequency 0 (i.e. "never seen in training")
    X_test[col + "_freq"] = X_test[col].map(freq_map).fillna(0.0)

X_train = X_train.drop(columns=freq_cols)
X_test = X_test.drop(columns=freq_cols)

X_train[["country_freq", "agent_freq"]].describe()

,country_freq,agent_freq
count,69916.000000,60181.000000
mean,0.138903,0.184992
std,0.122965,0.166396
min,0.000014,0.000017
25%,0.023643,0.016317
50%,0.101164,0.172962
75%,0.313233,0.383659
max,0.313233,0.383659


In [12]:
# One-hot encode the moderate-cardinality categoricals.
# Fit categories on the combined column so train/test end up with identical dummy columns.
X_train_ohe = pd.get_dummies(X_train, columns=onehot_cols, drop_first=False)
X_test_ohe = pd.get_dummies(X_test, columns=onehot_cols, drop_first=False)

# Align columns (in case a category appears only in one split) so both frames match exactly.
X_train_ohe, X_test_ohe = X_train_ohe.align(X_test_ohe, join="left", axis=1, fill_value=0)

print(f"Train shape after encoding: {X_train_ohe.shape}")
print(f"Test  shape after encoding: {X_test_ohe.shape}")
X_train_ohe.head(3)

Train shape after encoding: (69916, 81)
Test  shape after encoding: (17480, 81)


,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,total_nights,has_children_or_babies,country_freq,agent_freq,hotel_City Hotel,hotel_Resort Hotel,meal_BB,meal_FB,...,assigned_room_type_F,assigned_room_type_G,assigned_room_type_H,assigned_room_type_I,assigned_room_type_K,assigned_room_type_P,deposit_type_No Deposit,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Contract,customer_type_Group,customer_type_Transient,customer_type_Transient-Party,arrival_date_month_April,arrival_date_month_August,arrival_date_month_December,arrival_date_month_February,arrival_date_month_January,arrival_date_month_July,arrival_date_month_June,arrival_date_month_March,arrival_date_month_May,arrival_date_month_November,arrival_date_month_October,arrival_date_month_September
48772,22,2017,17,29,2,1,2,0.0,0,0,0,0,0,0,133.00,0,1,3,0,0.006436,0.383659,True,False,False,False,...,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False
65329,111,2016,29,13,0,3,2,0.0,0,0,0,0,0,0,72.25,0,0,3,0,0.101164,0.019857,True,False,True,False,...,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False
50281,422,2017,24,13,0,1,2,0.0,0,0,0,0,0,0,90.00,0,0,1,0,0.313233,0.002941,True,False,True,False,...,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False



## 7. Save the processed splits

We reattach the target column and write out the final train/test CSVs that
`03_train_test_eval.ipynb` will load directly.


In [13]:
train_out = X_train_ohe.copy()
train_out["is_canceled"] = y_train.values

test_out = X_test_ohe.copy()
test_out["is_canceled"] = y_test.values

train_out.to_csv("data/hotel_processed_train.csv", index=False)
test_out.to_csv("data/hotel_processed_test.csv", index=False)

print(f"Saved data/hotel_processed_train.csv  -> {train_out.shape}")
print(f"Saved data/hotel_processed_test.csv   -> {test_out.shape}")

Saved data/hotel_processed_train.csv  -> (69916, 82)
Saved data/hotel_processed_test.csv   -> (17480, 82)



## 8. Summary

- Proved, with a quick before/after model, why `reservation_status` and
  `reservation_status_date` are target leakage and must be dropped before modeling.
- Dropped `company` (94% missing), sentinel-filled `agent` with `"None"`, filled `country`
  with `"Unknown"`, and filled `children` with `0` — four different fixes for four different
  reasons.
- Engineered `total_nights` and `has_children_or_babies`.
- Skipped feature scaling entirely, because tree splits are invariant to monotonic
  transformations of a feature.
- One-hot encoded eight moderate-cardinality categoricals; frequency-encoded the
  high-cardinality `country` and `agent` columns instead of one-hot encoding them.
- Produced a stratified train/test split and saved `data/hotel_processed_train.csv` and
  `data/hotel_processed_test.csv` for the modeling notebook.
